#### 데이터 처리 파이프라인 
참고: Scikit-learn Pipelines 사용

In [1]:
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

설명: 미국 주 별 성별, 나이, 무게, 라벨(target)값 이용한 예측 모델 개발

In [2]:
data = {
	"state": ["CA", "WA", "CA", np.nan, "NV", "WA"],
	"gender": ["male", "female", "female", "male", np.nan, "female"],
	"age": [34, 29, 22, 44, 55, np.nan],
	"weight": [122, 150, 130, np.nan, 140, 175],
	"target": [0, 1, 0, 1, 0, 1]
}
df = pd.DataFrame(data)
X = df.drop("target", axis=1)
y = df["target"]

X

,state,gender,age,weight
0,CA,male,34.0,122.0
1,WA,female,29.0,150.0
2,CA,female,22.0,130.0
3,NaN,male,44.0,NaN
4,NV,NaN,55.0,140.0
5,WA,female,NaN,175.0


sklearn 모듈의 파이프라인 정의. 데이터 전처리 파이프라인 3개를 생성 후, 이들의 출력을 입력으로 받는 로지스틱 회귀 모델을 pipe 객체로 생성함. 
- numeric_preprocessor: 숫자형 데이터의 결측값을 평균으로 대체하고 표준화 처리.
- categorical_preprocessor: 범주형 데이터의 결측값을 'missing'으로 대체하고 원-핫 인코딩.
- preprocessor: 숫자형과 범주형 데이터를 각각의 파이프라인으로 전처리.
- pipe: 전처리된 데이터를 이용해 로지스틱 회귀 모델을 학습하는 전체 파이프라인.

In [3]:
numeric_preprocessor = Pipeline(
	steps=[
		("imputation_mean", SimpleImputer(missing_values=np.nan, strategy="mean")),
		("scaler", StandardScaler()),
	]
)

categorical_preprocessor = Pipeline(
	steps=[
		(
			"imputation_constant",
			SimpleImputer(fill_value="missing", strategy="constant"),
		),
		("onehot", OneHotEncoder(handle_unknown="ignore")),
	]
)

preprocessor = ColumnTransformer(
	[
		("categorical", categorical_preprocessor, ["state", "gender"]),
		("numerical", numeric_preprocessor, ["age", "weight"]),
	]
)

pipe = make_pipeline(preprocessor, LogisticRegression(max_iter=500))
pipe  # 파이프라인 구조 다이어그램 출력

,steps,"[('columntransformer', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('categorical', ...), ('numerical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


학습 로직 정의

In [4]:
# 데이터를 학습용과 테스트용으로 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 파이프라인을 학습 데이터에 맞춤
pipe.fit(X_train, y_train)

# 테스트 데이터에 대해 예측을 수행함
predictions = pipe.predict(X_test)

print("Actual values:", y_test.values)
print("Predicted values:", predictions)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 파이프라인을 학습 데이터에 맞춤
pipe.fit(X_train, y_train)

# 테스트 데이터에 대해 예측을 수행함
predictions = pipe.predict(X_test)

print("Actual values:", y_test.values)
print("Predicted values:", predictions)


Actual values: [0 1]
Predicted values: [0 1]
Actual values: [0 1]
Predicted values: [0 1]


학습된 로지스틱 회귀 모델의 정확도 계산 출력

In [5]:
accuracy = 0
correct = 0
for actual, predicted in zip(y_test.values, predictions):
	if actual == predicted:
		correct += 1
accuracy = correct / len(y_test)
print("Accuracy:", accuracy)

Accuracy: 1.0
